# Ablation — Basic Quantization Design Study

This notebook analyzes the 48 W&B runs from the quantization design study.

The sweep covers:

| Factor | Values |
|---|---|
| Bitwidth | `8`, `4` |
| Granularity | `per_tensor`, `per_channel` |
| Symmetry | `symmetric`, `asymmetric` |
| Calibration | `uncalibrated`, `percentile` |
| Seeds | `1337`, `42`, `123` |

Total expected runs: `2 × 2 × 2 × 2 × 3 = 48`.

The goal is to measure how basic design choices affect final round-trip validation BPB and compressed model size.

## Setup

This follows the same style as the previous ablation notebooks, but it includes fallback plotting style code so the notebook still runs even if `utils.py` is not available.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import wandb

# Optional project styling, compatible with previous notebooks.
try:
    import importlib, utils
    importlib.reload(utils)
    from utils import apply_style, get_deep_palette
    apply_style(force=True)
    PALETTE = get_deep_palette()
except Exception:
    plt.rcParams.update({
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })
    PALETTE = [
        "#4C78A8", "#F58518", "#54A24B", "#E45756",
        "#72B7B2", "#B279A2", "#FF9DA6", "#9D755D"
    ]

ENTITY  = "the-golfers"
PROJECT = "ml_ai_project"

api = wandb.Api()

BITS_ORDER = [8, 4]
GRAN_ORDER = ["per_tensor", "per_channel"]
SYM_ORDER = ["symmetric", "asymmetric"]
CAL_ORDER = ["uncalibrated", "percentile"]
SEED_ORDER = [1337, 42, 123]

DISPLAY_LABELS = {
    "per_tensor": "per-tensor",
    "per_channel": "per-channel",
    "symmetric": "symmetric",
    "asymmetric": "asymmetric",
    "uncalibrated": "uncalibrated",
    "percentile": "percentile-clipped",
}

OUT_DIR = Path("notebook_outputs/quant_design")
OUT_DIR.mkdir(parents=True, exist_ok=True)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: Enter your choice:

## Download runs from W&B

This cell uses the W&B API and saves a raw CSV similar to the export snippet from W&B.

It filters runs whose display name matches:

```text
quant_design_<bits>_<granularity>_<symmetry>_<calibration>_seed<seed>
```

If a run name appears more than once because a job was restarted, the notebook keeps the latest finished run.

In [ ]:
RUN_RE = re.compile(
    r"^quant_design_"
    r"(?P<bits>4|8)_"
    r"(?P<granularity>per_tensor|per_channel)_"
    r"(?P<symmetry>symmetric|asymmetric)_"
    r"(?P<calibration>uncalibrated|percentile)_"
    r"seed(?P<seed>1337|42|123)$"
)

expected_names = [
    f"quant_design_{bits}_{gran}_{sym}_{cal}_seed{seed}"
    for bits in BITS_ORDER
    for gran in GRAN_ORDER
    for sym in SYM_ORDER
    for cal in CAL_ORDER
    for seed in SEED_ORDER
]
expected_names = set(expected_names)

runs = list(api.runs(
    f"{ENTITY}/{PROJECT}",
    filters={"display_name": {"$regex": "^quant_design_(4|8)_(per_tensor|per_channel)_(symmetric|asymmetric)_(uncalibrated|percentile)_seed(1337|42|123)$"}},
))

print(f"Fetched {len(runs)} matching W&B run objects before de-duplication")

summary_list, config_list, name_list, state_list, created_at_list, run_id_list = [], [], [], [], [], []
for run in runs:
    summary_list.append(run.summary._json_dict)
    config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})
    name_list.append(run.name)
    state_list.append(run.state)
    created_at_list.append(run.created_at)
    run_id_list.append(run.id)

runs_df = pd.DataFrame({
    "summary": summary_list,
    "config": config_list,
    "name": name_list,
    "state": state_list,
    "created_at": created_at_list,
    "run_id": run_id_list,
})

runs_df.to_csv(OUT_DIR / "wandb_raw_quant_design_runs.csv", index=False)
runs_df.head()

## Parse run names and extract metrics

The main metric is `final_val_bpb`: this is the validation BPB after quantize → compress → reload/dequantize → evaluate.

The notebook also extracts:
- `final_val_loss`
- `quant_file_bytes`
- final pre-roundtrip `val_bpb` from history, when available
- `delta_bpb = final_val_bpb - fp32_val_bpb`, when available

In [ ]:
def first_present(mapping, keys, default=np.nan):
    for k in keys:
        if k in mapping and mapping[k] is not None:
            return mapping[k]
    return default

# Keep latest finished run if duplicates exist. If no finished run exists for a name, keep latest run anyway.
runs_sorted = sorted(runs, key=lambda r: str(r.created_at or ""))
latest_by_name = {}
for run in runs_sorted:
    if run.name not in latest_by_name:
        latest_by_name[run.name] = run
    elif run.state == "finished":
        latest_by_name[run.name] = run
    elif latest_by_name[run.name].state != "finished":
        latest_by_name[run.name] = run

selected_runs = list(latest_by_name.values())
print(f"Selected {len(selected_runs)} unique run names after de-duplication")

rows = []
for run in selected_runs:
    m = RUN_RE.match(run.name)
    if not m:
        continue
    info = m.groupdict()
    bits = int(info["bits"])
    seed = int(info["seed"])
    s = run.summary._json_dict
    cfg = {k: v for k, v in run.config.items() if not k.startswith("_")}

    # fp32/bf16 BPB before quantization, if the training history exists.
    fp32_bpb = np.nan
    try:
        hist = run.history(keys=["_step", "val_bpb"], pandas=True)
        if not hist.empty and "val_bpb" in hist:
            vals = hist["val_bpb"].dropna()
            if len(vals):
                fp32_bpb = float(vals.iloc[-1])
    except Exception as e:
        print(f"Could not fetch history for {run.name}: {e}")

    final_val_bpb = first_present(s, ["final_val_bpb", "val_bpb", "roundtrip_val_bpb"])
    final_val_loss = first_present(s, ["final_val_loss", "val_loss", "roundtrip_val_loss"])
    quant_file_bytes = first_present(s, ["quant_file_bytes", "compressed_model_bytes", "compressed_bytes", "artifact_bytes"])

    rows.append({
        "run_name": run.name,
        "run_id": run.id,
        "state": run.state,
        "created_at": run.created_at,
        "bits": bits,
        "granularity": info["granularity"],
        "symmetry": info["symmetry"],
        "calibration": info["calibration"],
        "seed": seed,
        "fp32_val_bpb": fp32_bpb,
        "final_val_bpb": final_val_bpb,
        "final_val_loss": final_val_loss,
        "quant_file_bytes": quant_file_bytes,
        "quant_file_MB": quant_file_bytes / 1e6 if pd.notna(quant_file_bytes) else np.nan,
        "matrix_quant_bits_cfg": cfg.get("matrix_quant_bits", cfg.get("MATRIX_QUANT_BITS", np.nan)),
        "quant_granularity_cfg": cfg.get("quant_granularity", cfg.get("QUANT_GRANULARITY", np.nan)),
        "quant_symmetry_cfg": cfg.get("quant_symmetry", cfg.get("QUANT_SYMMETRY", np.nan)),
        "quant_calibration_cfg": cfg.get("quant_calibration", cfg.get("QUANT_CALIBRATION", np.nan)),
    })

df = pd.DataFrame(rows)

# Ordering helpers
df["bits_order"] = df["bits"].map({v: i for i, v in enumerate(BITS_ORDER)})
df["gran_order"] = df["granularity"].map({v: i for i, v in enumerate(GRAN_ORDER)})
df["sym_order"] = df["symmetry"].map({v: i for i, v in enumerate(SYM_ORDER)})
df["cal_order"] = df["calibration"].map({v: i for i, v in enumerate(CAL_ORDER)})
df = df.sort_values(["bits_order", "gran_order", "sym_order", "cal_order", "seed"]).reset_index(drop=True)

# Quantization damage, when fp32 history is available.
df["delta_bpb"] = df["final_val_bpb"] - df["fp32_val_bpb"]

missing = sorted(expected_names - set(df["run_name"]))
extra = sorted(set(df["run_name"]) - expected_names)

print(f"Rows in parsed dataframe: {len(df)}")
print(f"Missing expected names: {len(missing)}")
if missing:
    print("\n".join(missing))
print(f"Extra names: {len(extra)}")
if extra:
    print("\n".join(extra))

bad_state = df[df["state"] != "finished"]
if not bad_state.empty:
    print("\nWarning: these selected runs are not marked finished:")
    print(bad_state[["run_name", "state", "created_at"]].to_string(index=False))

df.to_csv(OUT_DIR / "quant_design_raw_48_runs.csv", index=False)
df.head(12)

## Sanity checks

Each configuration should have 3 seeds.

In [ ]:
count_table = (
    df.groupby(["bits", "granularity", "symmetry", "calibration"])
      .size()
      .reset_index(name="n_seeds")
      .sort_values(["bits", "granularity", "symmetry", "calibration"])
)

print(count_table.to_string(index=False))

if len(df) != 48 or not (count_table["n_seeds"] == 3).all():
    print("\nWARNING: expected 48 total runs and 3 seeds per configuration. Check missing/duplicate/crashed runs above.")
else:
    print("\nOK: all 48 runs are present with 3 seeds per configuration.")

## Aggregate over seeds

This table is the main result: one row per quantization configuration.

In [ ]:
agg = (
    df.groupby(["bits", "granularity", "symmetry", "calibration"], as_index=False)
      .agg(
          n_seeds=("seed", "count"),
          final_val_bpb_mean=("final_val_bpb", "mean"),
          final_val_bpb_std=("final_val_bpb", "std"),
          final_val_loss_mean=("final_val_loss", "mean"),
          final_val_loss_std=("final_val_loss", "std"),
          fp32_val_bpb_mean=("fp32_val_bpb", "mean"),
          delta_bpb_mean=("delta_bpb", "mean"),
          delta_bpb_std=("delta_bpb", "std"),
          quant_file_MB_mean=("quant_file_MB", "mean"),
          quant_file_MB_std=("quant_file_MB", "std"),
      )
)

agg["bits_order"] = agg["bits"].map({v: i for i, v in enumerate(BITS_ORDER)})
agg["gran_order"] = agg["granularity"].map({v: i for i, v in enumerate(GRAN_ORDER)})
agg["sym_order"] = agg["symmetry"].map({v: i for i, v in enumerate(SYM_ORDER)})
agg["cal_order"] = agg["calibration"].map({v: i for i, v in enumerate(CAL_ORDER)})
agg = agg.sort_values(["bits_order", "gran_order", "sym_order", "cal_order"]).reset_index(drop=True)
agg = agg.drop(columns=["bits_order", "gran_order", "sym_order", "cal_order"])

agg["config"] = (
    agg["bits"].astype(str) + "-bit / " +
    agg["granularity"].map(DISPLAY_LABELS) + " / " +
    agg["symmetry"].map(DISPLAY_LABELS) + " / " +
    agg["calibration"].map(DISPLAY_LABELS)
)

agg.to_csv(OUT_DIR / "quant_design_aggregated_by_config.csv", index=False)

cols = [
    "config", "n_seeds",
    "final_val_bpb_mean", "final_val_bpb_std",
    "delta_bpb_mean", "delta_bpb_std",
    "quant_file_MB_mean", "quant_file_MB_std",
]
agg[cols]

## Best configurations

Lower `final_val_bpb_mean` is better.

In [ ]:
best = agg.sort_values("final_val_bpb_mean").reset_index(drop=True)

show = best[[
    "bits", "granularity", "symmetry", "calibration", "n_seeds",
    "final_val_bpb_mean", "final_val_bpb_std",
    "delta_bpb_mean", "quant_file_MB_mean"
]].copy()

for c in ["final_val_bpb_mean", "final_val_bpb_std", "delta_bpb_mean", "quant_file_MB_mean"]:
    show[c] = show[c].map(lambda x: f"{x:.6f}" if pd.notna(x) else "—")

show

## Pivot tables

These tables make the main comparisons easy to read.

In [ ]:
for bits in BITS_ORDER:
    print("=" * 80)
    print(f"{bits}-bit: mean final_val_bpb")
    print("=" * 80)
    sub = agg[agg["bits"] == bits].copy()
    pivot = sub.pivot_table(
        index=["granularity", "symmetry"],
        columns="calibration",
        values="final_val_bpb_mean",
    ).reindex(index=pd.MultiIndex.from_product([GRAN_ORDER, SYM_ORDER], names=["granularity", "symmetry"]), columns=CAL_ORDER)
    display(pivot)

    print("\n" + f"{bits}-bit: mean compressed file size (MB)")
    pivot_size = sub.pivot_table(
        index=["granularity", "symmetry"],
        columns="calibration",
        values="quant_file_MB_mean",
    ).reindex(index=pd.MultiIndex.from_product([GRAN_ORDER, SYM_ORDER], names=["granularity", "symmetry"]), columns=CAL_ORDER)
    display(pivot_size)

## Plot: final BPB by configuration

Each bar is the average over 3 seeds. Error bars show one standard deviation across seeds.

In [ ]:
plot_df = agg.copy()
plot_df["short_label"] = (
    plot_df["granularity"].map({"per_tensor": "tensor", "per_channel": "channel"}) + "\n" +
    plot_df["symmetry"].map({"symmetric": "sym", "asymmetric": "asym"}) + "\n" +
    plot_df["calibration"].map({"uncalibrated": "raw", "percentile": "pct"})
)

for bits in BITS_ORDER:
    sub = plot_df[plot_df["bits"] == bits].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(12, 5.5))
    x = np.arange(len(sub))
    colors = [PALETTE[i % len(PALETTE)] for i in range(len(sub))]

    ax.bar(
        x,
        sub["final_val_bpb_mean"],
        yerr=sub["final_val_bpb_std"],
        capsize=4,
        color=colors,
        width=0.65,
        zorder=3,
    )

    for i, val in enumerate(sub["final_val_bpb_mean"]):
        ax.text(i, val + 0.0004, f"{val:.4f}", ha="center", va="bottom", fontsize=9, rotation=0)

    ax.set_xticks(x)
    ax.set_xticklabels(sub["short_label"], fontsize=9)
    ax.set_ylabel("Final validation BPB ↓")
    ax.set_xlabel("Granularity / symmetry / calibration")
    ax.set_title(f"{bits}-bit quantization design study — final BPB")
    ymin = sub["final_val_bpb_mean"].min() - 0.003
    ymax = sub["final_val_bpb_mean"].max() + 0.006
    ax.set_ylim(ymin, ymax)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"final_bpb_{bits}bit_by_config.png", bbox_inches="tight")
    plt.show()

## Plot: compressed file size by configuration

This checks whether the design choices materially affect artifact size, not just BPB.

In [ ]:
for bits in BITS_ORDER:
    sub = plot_df[plot_df["bits"] == bits].reset_index(drop=True)
    if sub["quant_file_MB_mean"].isna().all():
        print(f"Skipping {bits}-bit size plot: quant_file_MB not found in summaries")
        continue

    fig, ax = plt.subplots(figsize=(12, 5.5))
    x = np.arange(len(sub))
    colors = [PALETTE[i % len(PALETTE)] for i in range(len(sub))]

    ax.bar(
        x,
        sub["quant_file_MB_mean"],
        yerr=sub["quant_file_MB_std"],
        capsize=4,
        color=colors,
        width=0.65,
        zorder=3,
    )

    for i, val in enumerate(sub["quant_file_MB_mean"]):
        if pd.notna(val):
            ax.text(i, val + 0.02, f"{val:.2f}", ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(sub["short_label"], fontsize=9)
    ax.set_ylabel("Compressed file size (MB)")
    ax.set_xlabel("Granularity / symmetry / calibration")
    ax.set_title(f"{bits}-bit quantization design study — compressed size")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"compressed_size_{bits}bit_by_config.png", bbox_inches="tight")
    plt.show()

## Main effects

This section averages across the other factors to estimate the overall effect of each design choice.

Interpretation: lower mean BPB is better.

In [ ]:
def main_effect_table(factor):
    tbl = (
        agg.groupby(["bits", factor], as_index=False)
           .agg(
               mean_bpb=("final_val_bpb_mean", "mean"),
               std_across_configs=("final_val_bpb_mean", "std"),
               mean_size_MB=("quant_file_MB_mean", "mean"),
           )
           .sort_values(["bits", factor])
    )
    return tbl

for factor in ["granularity", "symmetry", "calibration"]:
    print("=" * 80)
    print(f"Main effect: {factor}")
    print("=" * 80)
    display(main_effect_table(factor))

## Pairwise deltas

These tables compute paired differences while holding the other factors fixed.

Examples:
- `per_channel - per_tensor`: negative means per-channel is better.
- `asymmetric - symmetric`: negative means asymmetric is better.
- `percentile - uncalibrated`: negative means percentile clipping is better.

In [ ]:
def paired_delta(index_cols, factor_col, a, b, metric="final_val_bpb_mean"):
    piv = agg.pivot_table(index=index_cols, columns=factor_col, values=metric)
    piv[f"{b} - {a}"] = piv[b] - piv[a]
    return piv.reset_index()

print("Per-channel vs per-tensor: final_val_bpb_mean")
delta_gran = paired_delta(["bits", "symmetry", "calibration"], "granularity", "per_tensor", "per_channel")
display(delta_gran)

print("\nAsymmetric vs symmetric: final_val_bpb_mean")
delta_sym = paired_delta(["bits", "granularity", "calibration"], "symmetry", "symmetric", "asymmetric")
display(delta_sym)

print("\nPercentile vs uncalibrated: final_val_bpb_mean")
delta_cal = paired_delta(["bits", "granularity", "symmetry"], "calibration", "uncalibrated", "percentile")
display(delta_cal)

# Save deltas
delta_gran.to_csv(OUT_DIR / "delta_per_channel_minus_per_tensor.csv", index=False)
delta_sym.to_csv(OUT_DIR / "delta_asymmetric_minus_symmetric.csv", index=False)
delta_cal.to_csv(OUT_DIR / "delta_percentile_minus_uncalibrated.csv", index=False)

## Interaction plot

This plot helps show whether the best choice changes between 8-bit and 4-bit.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=False)

for ax, bits in zip(axes, BITS_ORDER):
    sub = agg[agg["bits"] == bits].copy()
    x_labels = ["sym/raw", "sym/pct", "asym/raw", "asym/pct"]
    x = np.arange(len(x_labels))

    for gran, color in zip(GRAN_ORDER, [PALETTE[0], PALETTE[2]]):
        vals = []
        errs = []
        for sym, cal in [("symmetric", "uncalibrated"), ("symmetric", "percentile"), ("asymmetric", "uncalibrated"), ("asymmetric", "percentile")]:
            row = sub[(sub["granularity"] == gran) & (sub["symmetry"] == sym) & (sub["calibration"] == cal)]
            vals.append(row["final_val_bpb_mean"].iloc[0])
            errs.append(row["final_val_bpb_std"].iloc[0])
        ax.errorbar(x, vals, yerr=errs, marker="o", linewidth=2, capsize=4, label=DISPLAY_LABELS[gran], color=color)

    ax.set_xticks(x)
    ax.set_xticklabels(x_labels)
    ax.set_ylabel("Final validation BPB ↓")
    ax.set_xlabel("Symmetry / calibration")
    ax.set_title(f"{bits}-bit interactions")
    ax.legend()

plt.tight_layout()
plt.savefig(OUT_DIR / "interaction_granularity_symmetry_calibration.png", bbox_inches="tight")
plt.show()

## Report-ready summary table

This table is formatted for copying into a report.

In [ ]:
report = agg[[
    "bits", "granularity", "symmetry", "calibration", "n_seeds",
    "final_val_bpb_mean", "final_val_bpb_std",
    "delta_bpb_mean", "quant_file_MB_mean"
]].copy()

report = report.sort_values(["bits", "granularity", "symmetry", "calibration"]).reset_index(drop=True)
report["granularity"] = report["granularity"].map(DISPLAY_LABELS)
report["symmetry"] = report["symmetry"].map(DISPLAY_LABELS)
report["calibration"] = report["calibration"].map(DISPLAY_LABELS)

report_display = report.copy()
report_display["final_val_bpb"] = report_display.apply(
    lambda r: f"{r['final_val_bpb_mean']:.4f} ± {r['final_val_bpb_std']:.4f}", axis=1
)
report_display["delta_bpb"] = report_display["delta_bpb_mean"].map(lambda x: f"{x:+.4f}" if pd.notna(x) else "—")
report_display["size_MB"] = report_display["quant_file_MB_mean"].map(lambda x: f"{x:.2f}" if pd.notna(x) else "—")

report_display = report_display[[
    "bits", "granularity", "symmetry", "calibration", "n_seeds", "final_val_bpb", "delta_bpb", "size_MB"
]]

report_display.to_csv(OUT_DIR / "report_ready_quant_design_table.csv", index=False)
report_display.to_markdown(OUT_DIR / "report_ready_quant_design_table.md", index=False)
report_display

## Auto-generated conclusion notes

These are not final prose, but they give you the numbers to discuss.

In [ ]:
print("Best overall configuration:")
best_row = agg.loc[agg["final_val_bpb_mean"].idxmin()]
print(
    f"  {int(best_row['bits'])}-bit, {DISPLAY_LABELS[best_row['granularity']]}, "
    f"{DISPLAY_LABELS[best_row['symmetry']]}, {DISPLAY_LABELS[best_row['calibration']]} "
    f"-> final_val_bpb={best_row['final_val_bpb_mean']:.6f} ± {best_row['final_val_bpb_std']:.6f}"
)

print("\nWorst overall configuration:")
worst_row = agg.loc[agg["final_val_bpb_mean"].idxmax()]
print(
    f"  {int(worst_row['bits'])}-bit, {DISPLAY_LABELS[worst_row['granularity']]}, "
    f"{DISPLAY_LABELS[worst_row['symmetry']]}, {DISPLAY_LABELS[worst_row['calibration']]} "
    f"-> final_val_bpb={worst_row['final_val_bpb_mean']:.6f} ± {worst_row['final_val_bpb_std']:.6f}"
)

print("\nAverage paired effects, negative is better:")
print(f"  per_channel - per_tensor: {delta_gran['per_channel - per_tensor'].mean():+.6f} BPB")
print(f"  asymmetric - symmetric:   {delta_sym['asymmetric - symmetric'].mean():+.6f} BPB")
print(f"  percentile - raw:         {delta_cal['percentile - uncalibrated'].mean():+.6f} BPB")

print("\nSuggested report wording template:")
print(
    "In the basic quantization design study, we evaluated bitwidth, granularity, symmetry, "
    "and clipping-based calibration over three seeds. The best configuration was "
    f"{int(best_row['bits'])}-bit {DISPLAY_LABELS[best_row['granularity']]} "
    f"{DISPLAY_LABELS[best_row['symmetry']]} with {DISPLAY_LABELS[best_row['calibration']]}, "
    f"achieving {best_row['final_val_bpb_mean']:.4f} BPB on average. "
    "The paired-delta tables above isolate the contribution of each design choice."
)